# 📊 Teacher vs Student Model Comparison

This notebook provides a comprehensive comparison between the **Teacher** and **Student** models after knowledge distillation training. 

## Objectives
- Load and evaluate both models on the same dataset
- Compare performance metrics (Accuracy, F1, Precision, Recall)
- Visualize confusion matrices and performance differences
- Analyze model compression benefits
- Generate a detailed comparison report

## Phases
1. **Setup & Configuration** - Import libraries and configure device
2. **Model Loading** - Load teacher and student models
3. **Dataset Preparation** - Load evaluation dataset
4. **Evaluation** - Run inference on both models
5. **Metrics Computation** - Calculate performance metrics
6. **Visualization** - Create comparison charts
7. **Reporting** - Generate comprehensive analysis

## 1️⃣ Setup and Configuration

Import all required libraries and configure the device for model inference.

In [ ]:
# Import required libraries
import sys
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

# Import project modules
from evaluation.model_comparison import ModelComparator
from data.dataloaders import get_imdb_dataloaders

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")

In [ ]:
# Configure device
if torch.backends.mps.is_available():
    device = "mps"
    print("🍎 Using Apple Silicon (MPS)")
elif torch.cuda.is_available():
    device = "cuda"
    print("🚀 Using CUDA")
else:
    device = "cpu"
    print("💻 Using CPU")

# Set experiment directory (UPDATE THIS TO YOUR LATEST EXPERIMENT)
EXPERIMENT_DIR = "../experiments/20250926T052444Z_ba9f0508"
TEACHER_PATH = f"{EXPERIMENT_DIR}/teacher_model"
STUDENT_PATH = f"{EXPERIMENT_DIR}/student_model"
OUTPUT_DIR = f"{EXPERIMENT_DIR}/comparison"

print(f"\n📁 Experiment Directory: {EXPERIMENT_DIR}")
print(f"👨‍🏫 Teacher Model: {TEACHER_PATH}")
print(f"👨‍🎓 Student Model: {STUDENT_PATH}")
print(f"💾 Output Directory: {OUTPUT_DIR}")

## 2️⃣ Load Teacher and Student Models

Load both models using the `ModelComparator` class which handles model loading, tokenizer setup, and device placement.

In [ ]:
# Initialize Model Comparator
print("🔄 Initializing Model Comparator...")
print("="*60)

comparator = ModelComparator(
    teacher_path=TEACHER_PATH,
    student_path=STUDENT_PATH,
    device=device,
    use_same_tokenizer=True  # Use student tokenizer for both models (recommended)
)

print("\n✅ Models loaded successfully!")
print(f"📊 Compression ratio: {comparator.compression_ratio:.2f}x")
print(f"📉 Size reduction: {(1 - 1/comparator.compression_ratio)*100:.1f}%")

## 3️⃣ Prepare Evaluation Dataset

Load the IMDB validation dataset that will be used to evaluate both models fairly.

In [ ]:
# Load evaluation dataset
print("📚 Loading evaluation dataset...")

train_loader, val_loader = get_imdb_dataloaders(
    train_path="../data/imdb_train.jsonl",
    val_path="../data/imdb_val.jsonl",
    tokenizer=comparator.tokenizer,
    batch_size=8,
    max_length=128
)

print("✅ Dataset loaded successfully!")
print(f"   Validation samples: {len(val_loader.dataset)}")
print(f"   Batch size: {val_loader.batch_size}")
print(f"   Number of batches: {len(val_loader)}")

## 4️⃣ Evaluate Both Models

Run the evaluation on both Teacher and Student models using the same dataset and conditions.

In [ ]:
# Run comparison
print("\n" + "="*60)
print("🎯 RUNNING TEACHER vs STUDENT COMPARISON")
print("="*60 + "\n")

teacher_results, student_results = comparator.compare_models(val_loader)

print("\n" + "="*60)
print("✅ EVALUATION COMPLETE!")
print("="*60)

## 5️⃣ Detailed Metrics Comparison

Let's examine the detailed metrics for both models side by side.

In [ ]:
# Display metrics comparison
import pandas as pd

metrics_data = {
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Avg Loss', 'Parameters'],
    'Teacher': [
        f"{teacher_results['accuracy']:.4f}",
        f"{teacher_results['precision']:.4f}",
        f"{teacher_results['recall']:.4f}",
        f"{teacher_results['f1']:.4f}",
        f"{teacher_results['loss']:.4f}",
        f"{teacher_results['num_parameters']:,}"
    ],
    'Student': [
        f"{student_results['accuracy']:.4f}",
        f"{student_results['precision']:.4f}",
        f"{student_results['recall']:.4f}",
        f"{student_results['f1']:.4f}",
        f"{student_results['loss']:.4f}",
        f"{student_results['num_parameters']:,}"
    ],
    'Difference': [
        f"{(teacher_results['accuracy'] - student_results['accuracy']):.4f}",
        f"{(teacher_results['precision'] - student_results['precision']):.4f}",
        f"{(teacher_results['recall'] - student_results['recall']):.4f}",
        f"{(teacher_results['f1'] - student_results['f1']):.4f}",
        f"{(teacher_results['loss'] - student_results['loss']):.4f}",
        f"{student_results['compression_ratio']:.2f}x smaller"
    ]
}

df_metrics = pd.DataFrame(metrics_data)
print("\n📊 Metrics Comparison Table:\n")
print(df_metrics.to_string(index=False))

# Calculate key statistics
acc_drop = (teacher_results['accuracy'] - student_results['accuracy']) * 100
f1_drop = (teacher_results['f1'] - student_results['f1']) * 100
compression = student_results['compression_ratio']

print("\n🎯 Key Insights:")
print(f"   • Accuracy drop: {acc_drop:.2f}%")
print(f"   • F1-Score drop: {f1_drop:.2f}%")
print(f"   • Model compression: {compression:.2f}x")
print(f"   • Size reduction: {(1 - 1/compression)*100:.1f}%")

## 6️⃣ Generate Visualizations

Create comprehensive visualizations comparing the models across different dimensions.

In [ ]:
# Generate all visualizations
print("📊 Generating visualizations...")

comparator.visualize_comparison(
    teacher_results,
    student_results,
    save_dir=OUTPUT_DIR,
    show_plots=False
)

print("\n✅ Visualizations generated successfully!")

In [ ]:
# Display visualizations inline
from IPython.display import Image, display

print("\n📈 Metrics Comparison:")
display(Image(filename=f"{OUTPUT_DIR}/metrics_comparison.png", width=800))

print("\n🎯 Confusion Matrices:")
display(Image(filename=f"{OUTPUT_DIR}/confusion_matrices_comparison.png", width=900))

print("\n📊 Per-Class Performance:")
display(Image(filename=f"{OUTPUT_DIR}/per_class_comparison.png", width=900))

print("\n⚡ Efficiency Analysis:")
display(Image(filename=f"{OUTPUT_DIR}/efficiency_comparison.png", width=700))

## 7️⃣ Save Results and Generate Report

Save the comparison results and generate a comprehensive markdown report.

In [ ]:
# Save results
comparator.save_results(
    teacher_results,
    student_results,
    save_dir=OUTPUT_DIR
)

# Generate report
comparator.generate_report(
    teacher_results,
    student_results,
    save_dir=OUTPUT_DIR
)

print("\n✅ All results and reports saved!")

## 8️⃣ Analyze Model Compression Benefits

Deep dive into the compression benefits and trade-offs.

In [ ]:
# Compression Analysis
print("🔍 COMPRESSION ANALYSIS\n" + "="*60)

# Calculate metrics
teacher_params = teacher_results['num_parameters']
student_params = student_results['num_parameters']
compression_ratio = teacher_params / student_params
size_reduction = (1 - 1/compression_ratio) * 100

teacher_acc = teacher_results['accuracy']
student_acc = student_results['accuracy']
accuracy_retention = (student_acc / teacher_acc) * 100

# Model sizes (approximate in MB)
param_size_bytes = 4  # 4 bytes per float32 parameter
teacher_size_mb = (teacher_params * param_size_bytes) / (1024**2)
student_size_mb = (student_params * param_size_bytes) / (1024**2)

print("\n📦 Model Size Analysis:")
print(f"   Teacher:  {teacher_size_mb:.1f} MB ({teacher_params:,} parameters)")
print(f"   Student:  {student_size_mb:.1f} MB ({student_params:,} parameters)")
print(f"   Saved:    {teacher_size_mb - student_size_mb:.1f} MB")
print(f"   Reduction: {size_reduction:.1f}%")

print("\n🎯 Performance Analysis:")
print(f"   Teacher Accuracy:  {teacher_acc:.4f}")
print(f"   Student Accuracy:  {student_acc:.4f}")
print(f"   Accuracy Retention: {accuracy_retention:.2f}%")
print(f"   Accuracy Drop:      {(teacher_acc - student_acc)*100:.2f}%")

print("\n⚡ Efficiency Metrics:")
print(f"   Compression Ratio:  {compression_ratio:.2f}x")
print(f"   Params per 1% Acc:  Teacher: {teacher_params/teacher_acc/100:,.0f}")
print(f"                       Student: {student_params/student_acc/100:,.0f}")

# Verdict
if abs(teacher_acc - student_acc) < 0.02:
    verdict = "✅ EXCELLENT"
    description = "Student maintains near-identical performance with significant compression!"
elif abs(teacher_acc - student_acc) < 0.05:
    verdict = "✔️ GOOD"
    description = "Student shows acceptable performance with good compression ratio."
else:
    verdict = "⚠️ FAIR"
    description = "Student shows noticeable performance drop. Consider retraining."

print(f"\n{verdict}: {description}")
print("="*60)

## 9️⃣ Confusion Matrix Analysis

Analyze the confusion matrices to understand where each model makes mistakes.

In [ ]:
# Analyze confusion matrices
print("🎯 CONFUSION MATRIX ANALYSIS\n" + "="*60)

teacher_cm = np.array(teacher_results['confusion_matrix'])
student_cm = np.array(student_results['confusion_matrix'])

print("\n👨‍🏫 Teacher Model Confusion Matrix:")
print(f"   True Negatives:  {teacher_cm[0, 0]:4d}  |  False Positives: {teacher_cm[0, 1]:4d}")
print(f"   False Negatives: {teacher_cm[1, 0]:4d}  |  True Positives:  {teacher_cm[1, 1]:4d}")

teacher_accuracy = (teacher_cm[0, 0] + teacher_cm[1, 1]) / teacher_cm.sum()
teacher_fp_rate = teacher_cm[0, 1] / (teacher_cm[0, 0] + teacher_cm[0, 1])
teacher_fn_rate = teacher_cm[1, 0] / (teacher_cm[1, 0] + teacher_cm[1, 1])

print(f"\n   Accuracy: {teacher_accuracy:.4f}")
print(f"   False Positive Rate: {teacher_fp_rate:.4f}")
print(f"   False Negative Rate: {teacher_fn_rate:.4f}")

print("\n👨‍🎓 Student Model Confusion Matrix:")
print(f"   True Negatives:  {student_cm[0, 0]:4d}  |  False Positives: {student_cm[0, 1]:4d}")
print(f"   False Negatives: {student_cm[1, 0]:4d}  |  True Positives:  {student_cm[1, 1]:4d}")

student_accuracy = (student_cm[0, 0] + student_cm[1, 1]) / student_cm.sum()
student_fp_rate = student_cm[0, 1] / (student_cm[0, 0] + student_cm[0, 1])
student_fn_rate = student_cm[1, 0] / (student_cm[1, 0] + student_cm[1, 1])

print(f"\n   Accuracy: {student_accuracy:.4f}")
print(f"   False Positive Rate: {student_fp_rate:.4f}")
print(f"   False Negative Rate: {student_fn_rate:.4f}")

# Comparison
print("\n📊 Comparison:")
print(f"   FP Rate Difference: {(student_fp_rate - teacher_fp_rate):.4f}")
print(f"   FN Rate Difference: {(student_fn_rate - teacher_fn_rate):.4f}")

if student_fn_rate > teacher_fn_rate * 1.5:
    print("   ⚠️  Student has significantly higher False Negative rate")
elif student_fp_rate > teacher_fp_rate * 1.5:
    print("   ⚠️  Student has significantly higher False Positive rate")
else:
    print("   ✅ Student maintains similar error distribution")

print("="*60)

## 🔟 Final Summary and Recommendations

Comprehensive summary with recommendations for deployment.

In [ ]:
# Final Summary
print("\n" + "="*60)
print("🎉 FINAL SUMMARY AND RECOMMENDATIONS")
print("="*60)

# Key metrics
acc_drop_pct = (teacher_acc - student_acc) * 100
f1_drop_pct = (teacher_results['f1'] - student_results['f1']) * 100

print("\n📊 Performance Summary:")
print(f"   Teacher: {teacher_acc:.4f} accuracy, {teacher_results['f1']:.4f} F1")
print(f"   Student: {student_acc:.4f} accuracy, {student_results['f1']:.4f} F1")
print(f"   Drop:    {acc_drop_pct:.2f}% accuracy, {f1_drop_pct:.2f}% F1")

print("\n💾 Model Size:")
print(f"   Compression: {compression_ratio:.2f}x smaller")
print(f"   Size saved: {teacher_size_mb - student_size_mb:.1f} MB")

print("\n🎯 Recommendations:")

if abs(acc_drop_pct) < 1.0:
    print("   ✅ DEPLOY STUDENT MODEL")
    print("      • Excellent accuracy retention (<1% drop)")
    print("      • Significant size reduction")
    print("      • Ideal for production deployment")
    print("      • Suitable for edge devices and mobile")
elif abs(acc_drop_pct) < 3.0:
    print("   ✔️ DEPLOY WITH CONFIDENCE")
    print("      • Good accuracy retention (<3% drop)")
    print("      • Good compression ratio")
    print("      • Suitable for most production scenarios")
    print("      • Consider A/B testing in production")
elif abs(acc_drop_pct) < 5.0:
    print("   ⚠️ DEPLOY WITH CAUTION")
    print("      • Moderate accuracy drop (3-5%)")
    print("      • Evaluate if compression justifies the drop")
    print("      • Consider use-case criticality")
    print("      • May need hyperparameter tuning")
else:
    print("   ❌ CONSIDER RETRAINING")
    print("      • Significant accuracy drop (>5%)")
    print("      • May need different architecture")
    print("      • Adjust distillation hyperparameters")
    print("      • Consider intermediate model size")

print(f"\n📁 All outputs saved to: {OUTPUT_DIR}")
print("\n✅ Comparison analysis complete!")
print("="*60)